In [38]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print(f"FAISS version: {faiss.__version__}")
print("✅ Libraries imported successfully!")

FAISS version: 1.13.2
✅ Libraries imported successfully!


In [39]:
# Sample documents
documents = [
    "Python is a versatile programming language used for web development and data science.",
    "Machine learning models require large amounts of training data to perform well.",
    "Neural networks are inspired by the structure of the human brain.",
    "Natural language processing enables computers to understand human language.",
    "Deep learning is a subset of machine learning using multi-layered neural networks.",
    "Data visualization helps communicate insights from complex datasets.",
    "Cloud computing provides on-demand access to computing resources.",
    "Cybersecurity protects systems and networks from digital attacks.",
    "Blockchain technology enables secure, decentralized transactions.",
    "Quantum computing uses quantum mechanics to solve complex problems."
]

print(f"Total documents: {len(documents)}")

Total documents: 10


In [40]:
# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
embeddings = model.encode(documents)

print(f"Generated {len(embeddings)} embeddings")
print(f"Each embedding has {embeddings.shape[1]} dimensions")
print(f"Embeddings shape: {embeddings.shape}")

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 12e9b348-cbfe-40d2-9ac9-907a0156db10)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Generated 10 embeddings
Each embedding has 384 dimensions
Embeddings shape: (10, 384)


In [41]:
# Get embedding dimension
dimension = embeddings.shape[1]

# Create FAISS index (IndexFlatL2 = exact search with L2 distance)
index = faiss.IndexFlatL2(dimension)

# Add embeddings to index
index.add(embeddings)

print(f"✅ FAISS index created!")
print(f"Total vectors in index: {index.ntotal}")

✅ FAISS index created!
Total vectors in index: 10


In [42]:
# Query
query = "What is artificial intelligence and machine learning?"

# Embed query
query_embedding = model.encode([query])

# Search: find top 3 most similar vectors
k = 3
distances, indices = index.search(query_embedding, k)

print(f"Query: {query}\n")
print(f"Top {k} results:\n")

for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{i}. (Distance: {distance:.4f})")
    print(f"   {documents[idx]}")
    print()

Query: What is artificial intelligence and machine learning?

Top 3 results:

1. (Distance: 0.9079)
   Deep learning is a subset of machine learning using multi-layered neural networks.

2. (Distance: 1.2202)
   Machine learning models require large amounts of training data to perform well.

3. (Distance: 1.2355)
   Natural language processing enables computers to understand human language.



In [43]:
# Normalize embeddings for cosine similarity
embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# Create index with inner product (equivalent to cosine for normalized vectors)
index_cosine = faiss.IndexFlatIP(dimension)
index_cosine.add(embeddings_normalized)

# Search with normalized query
query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding)
scores, indices = index_cosine.search(query_embedding_normalized, k=3)

print(f"Query: {query}\n")
print(f"Top {k} results with cosine similarity:\n")

for i, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
    print(f"{i}. (Similarity: {score:.4f})")
    print(f"   {documents[idx]}")
    print()

Query: What is artificial intelligence and machine learning?

Top 3 results with cosine similarity:

1. (Similarity: 0.5460)
   Deep learning is a subset of machine learning using multi-layered neural networks.

2. (Similarity: 0.3899)
   Machine learning models require large amounts of training data to perform well.

3. (Similarity: 0.3823)
   Natural language processing enables computers to understand human language.



In [44]:
# Save index to disk
faiss.write_index(index_cosine, "my_faiss_index.bin")
print("✅ Index saved to disk")

# Save documents separately (FAISS only stores vectors, not text)
import pickle
with open("documents.pkl", "wb") as f:
    pickle.dump(documents, f)
print("✅ Documents saved")

✅ Index saved to disk
✅ Documents saved


In [45]:
# Load index from disk
loaded_index = faiss.read_index("my_faiss_index.bin")
print(f"✅ Index loaded: {loaded_index.ntotal} vectors")

# Load documents
with open("documents.pkl", "rb") as f:
    loaded_documents = pickle.load(f)
print(f"✅ Documents loaded: {len(loaded_documents)} documents")

✅ Index loaded: 10 vectors
✅ Documents loaded: 10 documents


In [1]:
import chromadb

print(f"ChromaDB version: {chromadb.__version__}")
print("✅ ChromaDB imported successfully!")

ChromaDB version: 1.4.0
✅ ChromaDB imported successfully!


In [2]:
# Create Chroma client (persistent storage)
# Note: ChromaDB 0.4.0+ uses PersistentClient instead of Client(Settings(...))
client = chromadb.PersistentClient(path="./chroma_db")

# Create or get collection
collection = client.get_or_create_collection(
    name="my_documents",
    metadata={"description": "Sample document collection"}
)

print(f"✅ Collection created: {collection.name}")
print(f"Current count: {collection.count()} documents")
print(f"📁 Data persisted to: ./chroma_db/")

✅ Collection created: my_documents
Current count: 0 documents
📁 Data persisted to: ./chroma_db/
